# S&P 500 Market Performance Analytics

Track A: Data Analytics. This notebook reproduces preparation, analysis, selected-stock comparisons and validation for the 2014–2017 dataset.

In [ ]:
import pandas as pd,numpy as np, matplotlib.pyplot as plt, plotly.express as px
DATA_PATH = 'S&P 500 Stock Prices 2014-2017.csv'
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['symbol','date']).reset_index(drop=True)
df.head()

## Data inspection and quality checks

In [ ]:
print('Shape:', df.shape)
print('Symbols:', df.symbol.nunique())
print('Date range:', df.date.min(), df.date.max())
print(df.dtypes)
print('Missing values:\n',df.isna().sum())
print('Duplicate symbol-date:',df.duplicated(['symbol','date']).sum())
print('Zero volume:',(df.volume==0).sum())

## Feature engineering
Returns and moving averages are calculated separately by symbol after sorting chronologically. Missing intraday fields are retained rather than replaced with zero.

In [ ]:
df['year']=df.date.dt.year
df['month']=df.date.dt.month
df['quarter']=df.date.dt.quarter
df['daily_change']=df.groupby('symbol').close.diff()
df['daily_return']=df.groupby('symbol').close.pct_change()
df['trading_range']=df.high-df.low
df['MA5']=df.groupby('symbol').close.transform(lambda x:x.rolling(5,min_periods=5).mean())
df['MA20']=df.groupby('symbol').close.transform(lambda x:x.rolling(20,min_periods=20).mean())
df['MA50']=df.groupby('symbol').close.transform(lambda x:x.rolling(50,min_periods=50).mean())
df.head()

## Selected stock comparison
Five symbols with full 1,007 observation coverage were selected: AAPL, AMZN, GOOGL, JPM, MSFT.

In [ ]:
selected_symbols=['AAPL','AMZN','GOOGL','JPM','MSFT']
sel=df[df.symbol.isin(selected_symbols)].copy()
metrics=sel.groupby('symbol').agg(records=('close','count'),mean_close=('close','mean'),median_close=('close','median'),min_close=('close','min'),max_close=('close','max'),mean_return=('daily_return','mean'),volatility=('daily_return','std'),mean_volume=('volume','mean'),max_volume=('volume','max')).reset_index()
metrics['annualized_volatility']=metrics.volatility*np.sqrt(252)
metrics['period_return_pct']=sel.groupby('symbol').apply(lambda g: (g.close.iloc[-1]/g.close.iloc[0]-1)*100,include_groups=False).values
metrics

## Charts

In [ ]:
px.line(sel,x='date',y='close',color='symbol',title='Closing Price Trends').show()
px.line(sel,x='date',y='daily_return',color='symbol',title='Daily Returns').show()
px.bar(metrics,x='symbol',y='annualized_volatility',title='Annualized Volatility').show()
px.bar(metrics,x='symbol',y='mean_volume',title='Average Daily Volume').show()
px.bar(metrics,x='symbol',y='period_return_pct',title='Period Return').show()

## Full-dataset analysis

In [ ]:
max_close=df.loc[df.close.idxmax(),['symbol','date','close']]
min_close=df.loc[df.close.idxmin(),['symbol','date','close']]
period_returns=df.groupby('symbol').apply(lambda g:(g.close.iloc[-1]/g.close.iloc[0]-1)*100).sort_values(ascending=False)
volatility=df.groupby('symbol').daily_return.std().sort_values(ascending=False)
mean_volume=df.groupby('symbol').volume.mean().sort_values(ascending=False)
print('Highest closing observation:',max_close)
print('Lowest closing observation:',min_close)
print('Top period returns:',period_returns.head())
print('Lowest period returns:',period_returns.tail())
print('Highest return volatility:',volatility.head())
print('Highest mean volume:',mean_volume.head())

## Validation
Cross-check return formula and moving averages for a chosen symbol and a sample date. Note that this dataset has no adjusted-close column, so corporate-action-adjusted performance cannot be confirmed.

In [ ]:
sample=df[df.symbol=='AAPL'].copy()
manual=(sample.close.iloc[1]-sample.close.iloc[0])/sample.close.iloc[0]
assert np.isclose(sample.daily_return.iloc[1],manual)
manual_ma5=sample.close.iloc[:5].mean()
assert np.isclose(sample.MA5.iloc[4],manual_ma5)
print('AAPL second-observation return independently verified:',manual)
print('AAPL 5-day moving average verified:',manual_ma5)
print('Return formula and MA5 checks passed.')

## Interpretation and limitations

- Results are historical and cover 2014–2017 only.
- No adjusted close is supplied; returns may be affected by dividends, splits or other corporate actions.
- Missing open/high/low fields are not imputed.
- Incomplete stock histories mean start-to-end returns are not necessarily comparable for all symbols.
- Volatility is the standard deviation of daily percentage returns; annualized volatility uses √252.
- Findings are descriptive and not investment advice.